# RIPPLe — End-to-End Data Access Demo

Retrieve a real, pipeline-generated `calexp` and its source catalog from the
bundled `pipelines_check` demo Butler repo, then make a small cutout.

The demo products use legacy (DP0.2-style) dataset names, so we use
`data_release="dp02"`. For real DP1 data, drop that argument (DP1 is the default
and resolves `get_calexp` to the `visit_image` dataset type).

In [1]:
import os
from lsst.daf.butler import Butler
from ripple.data_access import ButlerClient, ButlerConfig

# Works whether the kernel cwd is the repo root or the notebooks/ folder
REPO = "data/pipelines_check-29.1.1/DATA_REPO"
if not os.path.exists(REPO):
    REPO = os.path.join("..", REPO)
COLL = "demo_collection"

config = ButlerConfig(repo_path=REPO, collections=[COLL], data_release="dp02")
client = ButlerClient(config=config)
print("ButlerClient ready; data_release =", client.data_release)
print("resolved 'calexp' dataset type =", client._dt("calexp"))

ButlerClient ready; data_release = dp02
resolved 'calexp' dataset type = calexp


In [2]:
# Discover a real (visit, detector) present in the demo collection
butler = Butler(REPO, collections=[COLL])
ref = next(iter(butler.registry.queryDatasets("calexp", collections=[COLL])))
req = dict(ref.dataId.required)
visit, detector = int(req["visit"]), int(req["detector"])
print("discovered dataId:", {"visit": visit, "detector": detector})

discovered dataId: {'visit': 903342, 'detector': 10}


In [3]:
# Retrieve the calibrated exposure
exposure = client.get_calexp(visit=visit, detector=detector)
print("type:", type(exposure).__name__)
print("dimensions:", exposure.getDimensions())

type: ExposureF
dimensions: (2048, 4176)


In [4]:
# Retrieve the per-detector source catalog
catalog = client.get_source_catalog(visit=visit, detector=detector)
print("source catalog rows:", len(catalog))

source catalog rows: 1337


In [5]:
# Make a 64x64 cutout from the image center and display it
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

img = exposure.getImage().getArray()
cy, cx = img.shape[0] // 2, img.shape[1] // 2
cutout = img[cy - 32:cy + 32, cx - 32:cx + 32]
finite = cutout[np.isfinite(cutout)]
vmin, vmax = np.percentile(finite, [1, 99])
plt.figure(figsize=(4, 4))
plt.imshow(cutout, origin="lower", vmin=vmin, vmax=vmax, cmap="gray")
plt.title(f"calexp center 64x64 (visit={visit}, det={detector})")
plt.axis("off")
print("cutout shape:", cutout.shape)

cutout shape: (64, 64)
